In [1]:

import requests
import hashlib
import zipfile
import json

from pathlib import Path
from datetime import datetime, timezone

print("=== INGESTION INSEE POPULATION 200M ===")

URL = (
    "https://www.insee.fr/fr/statistiques/"
    "fichier/8735162/"
    "Filosofi2021_carreaux_200m_csv.zip"
)

DEST = Path(
    "/lakehouse/default/Files/bronze/"
    "insee_population/filosofi2021"
)

DEST.mkdir(parents=True, exist_ok=True)

FILE = DEST / "Filosofi2021_carreaux_200m_csv.zip"
TEMP = DEST / "download.part"

if not FILE.exists() or not zipfile.is_zipfile(FILE):

    print("Téléchargement des données INSEE...")

    h = hashlib.sha256()
    total = 0

    with requests.get(
        URL, stream=True, timeout=(30, 120)
    ) as response:

        response.raise_for_status()

        with open(TEMP, "wb") as f:
            for chunk in response.iter_content(
                chunk_size=1024 * 1024
            ):
                if chunk:
                    f.write(chunk)
                    h.update(chunk)
                    total += len(chunk)

    if total < 1024 * 1024:
        raise ValueError("Téléchargement trop petit")

    if not zipfile.is_zipfile(TEMP):
        raise ValueError("Fichier téléchargé non ZIP")

    with zipfile.ZipFile(TEMP) as z:
        bad = z.testzip()
        if bad is not None:
            raise ValueError(
                f"Archive corrompue : {bad}"
            )

    TEMP.replace(FILE)

else:
    print("Archive déjà présente, réutilisée.")

# Calculer le checksum local
sha256 = hashlib.sha256()

with open(FILE, "rb") as f:
    for chunk in iter(
        lambda: f.read(1024 * 1024), b""
    ):
        sha256.update(chunk)

manifest = {
    "source": URL,
    "filename": FILE.name,
    "sha256": sha256.hexdigest(),
    "size_bytes": FILE.stat().st_size,
    "checked_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "reference_year": 2021
}

(DEST / "provenance.json").write_text(
    json.dumps(manifest, indent=2),
    encoding="utf-8"
)

print("\n=== IMPORT BRONZE TERMINÉ ===")
print("Fichier :", FILE)
print(
    "Taille (Mo) :",
    round(FILE.stat().st_size / 1048576, 2)
)
print("SHA256 :", sha256.hexdigest())


In [2]:

import csv
import io
import zipfile

print("=== EXPLORATION INSEE ===")

with zipfile.ZipFile(FILE) as archive:

    files = [
        x for x in archive.namelist()
        if not x.endswith("/")
    ]

    print("Fichiers dans l'archive :")

    for f in files:
        print("-", f)

    csv_files = [
        f for f in files
        if f.lower().endswith(".csv")
    ]

    print("\nNombre de CSV :", len(csv_files))

    for filename in csv_files:
        print("\n=== FICHIER ===")
        print(filename)

        with archive.open(filename) as raw:
            text = io.TextIOWrapper(
                raw, encoding="utf-8-sig",
                errors="replace"
            )

            sample = text.read(8192)

            try:
                delimiter = csv.Sniffer().sniff(
                    sample, delimiters=";,\t"
                ).delimiter
            except csv.Error:
                delimiter = ";"

            header = next(
                csv.reader(
                    io.StringIO(sample),
                    delimiter=delimiter
                )
            )

            print("Séparateur :", repr(delimiter))
            print("Nombre de colonnes :", len(header))
            print("Colonnes :", header)


In [3]:

import zipfile
import shutil
from pathlib import Path

from pyspark.sql import functions as F
from functools import reduce

print("=== CRÉATION SILVER POPULATION FRANCE ===")

# ==========================================
# 1. CONFIGURATION
# ==========================================

BASE = Path(
    "/lakehouse/default/Files/bronze/"
    "insee_population/filosofi2021"
)

ZIP_PATH = BASE / "Filosofi2021_carreaux_200m_csv.zip"
EXTRACT_DIR = BASE / "extracted"

EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

TERRITORIES = {
    "carreaux_200m_met.csv": "METROPOLE",
    "carreaux_200m_mart.csv": "MARTINIQUE",
    "carreaux_200m_reun.csv": "REUNION"
}

REQUIRED_COLUMNS = {
    "idcar_200m",
    "idcar_1km",
    "idcar_nat",
    "i_est_200",
    "i_est_1km",
    "lcog_geo",
    "ind"
}

# ==========================================
# 2. EXTRAIRE LES CSV
# ==========================================

with zipfile.ZipFile(ZIP_PATH) as archive:

    for filename in TERRITORIES:

        destination = EXTRACT_DIR / filename

        if destination.exists() and destination.stat().st_size > 0:
            print("Déjà extrait :", filename)
            continue

        print("Extraction :", filename)

        matches = [
            n for n in archive.namelist()
            if Path(n).name == filename
        ]

        if len(matches) != 1:
            raise ValueError(
                f"Fichier absent ou ambigu : {filename}"
            )

        temp = destination.with_suffix(".csv.part")

        with archive.open(matches[0]) as source:
            with open(temp, "wb") as target:
                shutil.copyfileobj(
                    source, target,
                    length=8 * 1024 * 1024
                )

        temp.replace(destination)

print("\nExtraction terminée.")

# ==========================================
# 3. LIRE LES DONNÉES AVEC PYSPARK
# ==========================================

dataframes = []

for filename, territory in TERRITORIES.items():

    print("\nLecture :", territory)

    spark_path = (
        "Files/bronze/insee_population/"
        "filosofi2021/extracted/" + filename
    )

    df = (
        spark.read
        .option("header", "true")
        .option("sep", ",")
        .option("encoding", "UTF-8")
        .option("inferSchema", "false")
        .option("mode", "FAILFAST")
        .csv(spark_path)
    )

    missing = REQUIRED_COLUMNS - set(df.columns)

    if missing:
        raise ValueError(
            f"Colonnes manquantes dans {filename}: {missing}"
        )

    # Toutes les valeurs brutes restent conservées.
    df = (
        df.withColumn(
            "territory", F.lit(territory)
        )
        .withColumn(
            "source", F.lit("INSEE_FILOSOFI")
        )
        .withColumn(
            "reference_year", F.lit(2021)
        )
        .withColumn(
            "source_file", F.lit(filename)
        )
    )

    dataframes.append(df)

# ==========================================
# 4. FUSIONNER LES TROIS TERRITOIRES
# ==========================================

population = reduce(
    lambda a, b: a.unionByName(b),
    dataframes
)

# ==========================================
# 5. CONVERTIR LES COLONNES ESSENTIELLES
# ==========================================

population = (
    population
    .withColumn(
        "population",
        F.expr(
            "try_cast(replace(trim(ind), ',', '.') as double)"
        )
    )
    .withColumn(
        "is_imputed",
        F.expr("try_cast(i_est_200 as int)")
    )
    .withColumn(
        "is_imputed_1km",
        F.expr("try_cast(i_est_1km as int)")
    )
    .withColumn(
        "processed_at_utc",
        F.current_timestamp()
    )
)

# On signale les anomalies sans supprimer les données.
population = population.withColumn(
    "quality_valid",
    F.col("idcar_200m").isNotNull()
    & (F.trim(F.col("idcar_200m")) != "")
    & F.col("population").isNotNull()
    & (F.col("population") >= 0)
    & F.col("is_imputed").isin(0, 1)
)

# ==========================================
# 6. CONTRÔLES DE QUALITÉ
# ==========================================

total = population.count()

invalid = population.filter(
    ~F.col("quality_valid")
).count()

duplicates = (
    population
    .groupBy("territory", "idcar_200m")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("\n=== QUALITÉ DES DONNÉES ===")
print("Carreaux chargés :", total)
print("Lignes à vérifier :", invalid)
print("Identifiants dupliqués :", duplicates)

if duplicates:
    raise ValueError(
        "Identifiants de carreaux dupliqués : "
        "vérification nécessaire avant écriture"
    )

# ==========================================
# 7. ENREGISTRER DANS SILVER
# ==========================================

(
    population.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.france_population_200m")
)

print("\nTABLE CRÉÉE : silver.france_population_200m")

# ==========================================
# 8. STATISTIQUES NATIONALES
# ==========================================

result = spark.table("silver.france_population_200m")

print("\n=== RÉPARTITION PAR TERRITOIRE ===")

display(
    result.groupBy("territory")
    .agg(
        F.count("*").alias("nombre_carreaux"),
        F.round(
            F.sum(
                F.when(
                    F.col("quality_valid"),
                    F.col("population")
                )
            ), 0
        ).alias("population_reference"),
        F.sum(
            F.when(
                F.col("is_imputed") == 1, 1
            ).otherwise(0)
        ).alias("carreaux_imputes"),
        F.sum(
            F.when(
                ~F.col("quality_valid"), 1
            ).otherwise(0)
        ).alias("lignes_a_verifier")
    )
    .orderBy("territory")
)

print("\n=== APERÇU DES CARREAUX ===")

display(
    result.select(
        "idcar_200m",
        "territory",
        "population",
        "is_imputed",
        "quality_valid"
    ).limit(20)
)

print("\n=== RÉPARTITION IMPUTATION ===")

display(
    result.groupBy(
        "territory", "is_imputed"
    )
    .count()
    .orderBy("territory", "is_imputed")
)

print("\nTRAITEMENT TERMINÉ")


In [5]:
%pip install pyproj

In [6]:
import pyproj
print("pyproj :", pyproj.__version__)

In [7]:

import numpy as np
import pandas as pd

from pyproj import Transformer
from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark import StorageLevel

print("=== GEOLOCALISATION INSEE 200M ===")

source = spark.table(
    "silver.france_population_200m"
)

# 1. Lire les coordonnées des identifiants INSPIRE
pattern = (
    r"^CRS([0-9]+)RES200mN(-?[0-9]+)E(-?[0-9]+)$"
)

matches = F.col("idcar_200m").rlike(pattern)

df = (
    source.select(
        "idcar_200m",
        "territory",
        "population",
        "is_imputed",
        "reference_year"
    )
    .withColumn(
        "epsg",
        F.when(
            matches,
            F.regexp_extract(
                "idcar_200m", pattern, 1
            ).cast("int")
        )
    )
    .withColumn(
        "y_min",
        F.when(
            matches,
            F.regexp_extract(
                "idcar_200m", pattern, 2
            ).cast("double")
        )
    )
    .withColumn(
        "x_min",
        F.when(
            matches,
            F.regexp_extract(
                "idcar_200m", pattern, 3
            ).cast("double")
        )
    )
    .withColumn(
        "center_x_m", F.col("x_min") + 100.0
    )
    .withColumn(
        "center_y_m", F.col("y_min") + 100.0
    )
)

# 2. Vérifier la projection de chaque territoire
valid_crs = (
    (
        (F.col("territory") == "METROPOLE")
        & (F.col("epsg") == 3035)
    )
    | (
        (F.col("territory") == "MARTINIQUE")
        & (F.col("epsg") == 5490)
    )
    | (
        (F.col("territory") == "REUNION")
        & (F.col("epsg") == 2975)
    )
)

invalid_crs = df.filter(~valid_crs | valid_crs.isNull())

print("\n=== PROJECTIONS INSEE ===")

display(
    df.groupBy("territory", "epsg")
    .count()
    .orderBy("territory")
)

bad_count = invalid_crs.count()
print("Identifiants/projections invalides :", bad_count)

if bad_count:
    display(invalid_crs.limit(10))
    raise ValueError(
        "Projections inattendues : traitement interrompu"
    )

# 3. Conversion vectorisée vers WGS84
@pandas_udf(
    "struct<longitude:double,latitude:double>"
)
def project_wgs84(
    xs: pd.Series,
    ys: pd.Series,
    codes: pd.Series
) -> pd.DataFrame:

    x = xs.to_numpy(dtype="float64")
    y = ys.to_numpy(dtype="float64")
    crs = codes.to_numpy(dtype="int32")

    lon_out = np.full(len(x), np.nan)
    lat_out = np.full(len(x), np.nan)

    for code in np.unique(crs):
        mask = crs == code

        transformer = Transformer.from_crs(
            f"EPSG:{int(code)}",
            "EPSG:4326",
            always_xy=True
        )

        lon, lat = transformer.transform(
            x[mask], y[mask]
        )

        lon_out[mask] = lon
        lat_out[mask] = lat

    return pd.DataFrame({
        "longitude": lon_out,
        "latitude": lat_out
    })

geo = (
    df.withColumn(
        "_wgs84",
        project_wgs84(
            "center_x_m", "center_y_m", "epsg"
        )
    )
    .withColumn(
        "longitude", F.col("_wgs84.longitude")
    )
    .withColumn(
        "latitude", F.col("_wgs84.latitude")
    )
    .drop("_wgs84", "x_min", "y_min")
)

# 4. Vérifier que les points tombent
# dans des zones géographiques cohérentes
valid_location = (
    (
        (F.col("territory") == "METROPOLE")
        & F.col("longitude").between(-6.0, 11.0)
        & F.col("latitude").between(41.0, 52.0)
    )
    | (
        (F.col("territory") == "MARTINIQUE")
        & F.col("longitude").between(-62.0, -60.0)
        & F.col("latitude").between(14.0, 16.0)
    )
    | (
        (F.col("territory") == "REUNION")
        & F.col("longitude").between(54.0, 56.5)
        & F.col("latitude").between(-22.0, -20.0)
    )
)

geo = geo.withColumn(
    "geo_valid",
    F.coalesce(valid_location, F.lit(False))
)

geo = geo.persist(StorageLevel.MEMORY_AND_DISK)

try:
    invalid_geo = geo.filter(
        ~F.col("geo_valid")
    ).count()

    print("\nPoints géographiques invalides :", invalid_geo)

    if invalid_geo:
        display(
            geo.filter(~F.col("geo_valid")).limit(10)
        )
        raise ValueError(
            "Certaines coordonnées sont hors territoire"
        )

    # 5. Enregistrer dans Silver
    (
        geo.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(
            "silver.france_population_200m_geocoded"
        )
    )

finally:
    geo.unpersist()

# 6. Contrôle final
result = spark.table(
    "silver.france_population_200m_geocoded"
)

print("\n=== BILAN GEOGRAPHIQUE ===")

display(
    result.groupBy("territory", "epsg")
    .agg(
        F.count("*").alias("carreaux"),
        F.round(
            F.sum("population"), 0
        ).alias("population")
    )
    .orderBy("territory")
)

print("\n=== EXEMPLES GPS ===")

display(
    result.select(
        "idcar_200m",
        "territory",
        "latitude",
        "longitude",
        "population",
        "is_imputed"
    ).limit(15)
)

print("\nTABLE CREEE :")
print("silver.france_population_200m_geocoded")


In [8]:
%pip install h3

In [9]:
import h3

print("Version H3 :", h3.__version__)

print(
    "Test H3 Strasbourg :",
    h3.latlng_to_cell(48.5734, 7.7521, 8)
)

In [10]:

import pandas as pd
import h3

from pyspark.sql import functions as F
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType

print("=== AGRÉGATION POPULATION H3 FRANCE ===")

# ==========================================
# 1. LECTURE DES CARREAUX GÉOLOCALISÉS
# ==========================================

population = (
    spark.table(
        "silver.france_population_200m_geocoded"
    )
    .filter(F.col("geo_valid") == True)
    .select(
        "idcar_200m",
        "territory",
        "population",
        "is_imputed",
        "latitude",
        "longitude"
    )
)

# ==========================================
# 2. INDEXATION H3 RÉSOLUTION 8
# ==========================================

@pandas_udf("string")
def latlon_to_h3_r8(
    latitudes: pd.Series,
    longitudes: pd.Series
) -> pd.Series:

    return pd.Series([
        h3.latlng_to_cell(
            float(lat), float(lon), 8
        )
        for lat, lon in zip(
            latitudes, longitudes
        )
    ])

indexed = population.withColumn(
    "h3_cell",
    latlon_to_h3_r8(
        F.col("latitude"),
        F.col("longitude")
    )
)

# ==========================================
# 3. AGRÉGER LA POPULATION
# ==========================================

h3_r8 = (
    indexed.groupBy("territory", "h3_cell")
    .agg(
        F.sum("population").alias("population"),
        F.count("*").alias("nombre_carreaux_200m"),
        F.sum(
            F.when(
                F.col("is_imputed") == 1, 1
            ).otherwise(0)
        ).alias("carreaux_imputes"),
        F.sum(
            F.when(
                F.col("is_imputed") == 1,
                F.col("population")
            ).otherwise(0.0)
        ).alias("population_carreaux_imputes")
    )
    .withColumn("h3_resolution", F.lit(8))
    .withColumn("reference_year", F.lit(2021))
)

# ==========================================
# 4. ENREGISTRER H3 R8
# ==========================================

(
    h3_r8.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.france_population_h3_r8")
)

print("Table H3 R8 créée.")

# ==========================================
# 5. CRÉER H3 RÉSOLUTION 7
# ==========================================

@F.udf(returnType=StringType())
def h3_to_parent_r7(cell):
    if cell is None:
        return None
    return h3.cell_to_parent(cell, 7)

h3_r8_saved = spark.table(
    "silver.france_population_h3_r8"
)

h3_r7 = (
    h3_r8_saved.withColumn(
        "h3_cell",
        h3_to_parent_r7(F.col("h3_cell"))
    )
    .groupBy("territory", "h3_cell")
    .agg(
        F.sum("population").alias("population"),
        F.sum("nombre_carreaux_200m").alias(
            "nombre_carreaux_200m"
        ),
        F.sum("carreaux_imputes").alias(
            "carreaux_imputes"
        ),
        F.sum("population_carreaux_imputes").alias(
            "population_carreaux_imputes"
        )
    )
    .withColumn("h3_resolution", F.lit(7))
    .withColumn("reference_year", F.lit(2021))
)

(
    h3_r7.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.france_population_h3_r7")
)

print("Table H3 R7 créée.")

# ==========================================
# 6. CONTRÔLES
# ==========================================

print("\n=== BILAN H3 RÉSOLUTION 7 ===")

display(
    spark.table("silver.france_population_h3_r7")
    .groupBy("territory")
    .agg(
        F.count("*").alias("hexagones"),
        F.round(
            F.sum("population"), 0
        ).alias("population")
    )
    .orderBy("territory")
)

print("\n=== BILAN H3 RÉSOLUTION 8 ===")

display(
    spark.table("silver.france_population_h3_r8")
    .groupBy("territory")
    .agg(
        F.count("*").alias("hexagones"),
        F.round(
            F.sum("population"), 0
        ).alias("population")
    )
    .orderBy("territory")
)

# Vérifier que l'agrégation ne perd pas
# de population.
source_total = population.agg(
    F.sum("population")
).first()[0]

for resolution in [7, 8]:

    total = spark.table(
        f"silver.france_population_h3_r{resolution}"
    ).agg(F.sum("population")).first()[0]

    difference = abs(total - source_total)

    print(
        f"H3 R{resolution} :",
        f"population = {total:,.1f}",
        f"| écart = {difference:.6f}"
    )

    if difference > 1:
        raise ValueError(
            f"Perte de population en H3 R{resolution}"
        )

print("\n=== TABLES CRÉÉES ===")
print("silver.france_population_h3_r7")
print("silver.france_population_h3_r8")


In [11]:
%pip install folium branca

In [13]:

import h3
import folium
import pandas as pd
import numpy as np

from html import escape
from pathlib import Path
from branca.colormap import linear
from folium.plugins import MarkerCluster
from pyspark.sql import functions as F

print("=== PREMIÈRE CARTE NATIONALE ===")

# =========================================
# 1. CONFIGURATION
# =========================================

OUT = Path(
    "/lakehouse/default/Files/gold/maps"
)
OUT.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "METROPOLE": {
        "center": [46.6, 2.2],
        "zoom": 6,
        "filename": "mobilite_metropole.html"
    },
    "MARTINIQUE": {
        "center": [14.65, -61.0],
        "zoom": 10,
        "filename": "mobilite_martinique.html"
    },
    "REUNION": {
        "center": [-21.12, 55.54],
        "zoom": 10,
        "filename": "mobilite_reunion.html"
    }
}

# =========================================
# 2. AGRÉGATION H3 POUR VISUALISATION
# =========================================

@F.udf("string")
def visual_cell(cell, territory):
    if cell is None:
        return None

    if territory == "METROPOLE":
        return h3.cell_to_parent(cell, 5)

    return cell

population = (
    spark.table("silver.france_population_h3_r7")
    .withColumn(
        "map_h3",
        visual_cell("h3_cell", "territory")
    )
    .groupBy("territory", "map_h3")
    .agg(
        F.sum("population").alias("population"),
        F.sum("nombre_carreaux_200m").alias(
            "nombre_carreaux"
        )
    )
)

print("Agrégation terminée.")

# =========================================
# 3. CHARGER LES STATIONS CITIZ
# =========================================

citiz = (
    spark.table("silver.france_citiz_stations")
    .select(
        "station_key",
        "station_name",
        "operator",
        "latitude",
        "longitude"
    )
    .toPandas()
)

citiz["latitude"] = pd.to_numeric(
    citiz["latitude"], errors="coerce"
)
citiz["longitude"] = pd.to_numeric(
    citiz["longitude"], errors="coerce"
)

citiz = citiz.dropna(
    subset=["latitude", "longitude"]
)

def territory_for_point(lat, lon):
    if 41 <= lat <= 52 and -6 <= lon <= 11:
        return "METROPOLE"

    if 14 <= lat <= 16 and -62 <= lon <= -60:
        return "MARTINIQUE"

    if -22 <= lat <= -20 and 54 <= lon <= 57:
        return "REUNION"

    return "OUTSIDE"

citiz["territory"] = citiz.apply(
    lambda row: territory_for_point(
        row["latitude"], row["longitude"]
    ),
    axis=1
)

# =========================================
# 4. GÉNÉRATION DES CARTES
# =========================================

maps = {}

for territory, cfg in CONFIG.items():

    print("\nCréation :", territory)

    pop = (
        population
        .filter(F.col("territory") == territory)
        .toPandas()
    )

    stations = citiz[
        citiz["territory"] == territory
    ]

    m = folium.Map(
        location=cfg["center"],
        zoom_start=cfg["zoom"],
tiles="OpenStreetMap",
        prefer_canvas=True,
        control_scale=True
    )

    # Couleurs adaptées à chaque territoire
    upper = max(
        float(pop["population"].quantile(0.95)),
        1.0
    )

    colors = linear.YlOrRd_09.scale(0, upper)
    colors.caption = (
        "Population INSEE 2021 par hexagone "
        "(saturation au percentile 95)"
    )

    features = []

    for row in pop.itertuples():

        boundary = h3.cell_to_boundary(
            row.map_h3
        )

        coordinates = [
            [lon, lat]
            for lat, lon in boundary
        ]

        coordinates.append(coordinates[0])

        features.append({
            "type": "Feature",
            "geometry": {
                "type": "Polygon",
                "coordinates": [coordinates]
            },
            "properties": {
                "population": round(
                    float(row.population), 1
                ),
                "carreaux": int(row.nombre_carreaux),
                "h3": row.map_h3
            }
        })

    population_layer = folium.FeatureGroup(
        name="Population INSEE 2021",
        show=True
    )

    folium.GeoJson(
        {
            "type": "FeatureCollection",
            "features": features
        },
        name="Hexagones population",
        style_function=lambda feature: {
            "fillColor": colors(
                min(
                    feature["properties"]["population"],
                    upper
                )
            ),
            "fillOpacity": 0.65,
            "color": "#777777",
            "weight": 0.2
        },
        tooltip=folium.GeoJsonTooltip(
            fields=["population", "carreaux"],
            aliases=[
                "Population du groupe de carreaux :",
                "Carreaux INSEE :"
            ],
            localize=True
        )
    ).add_to(population_layer)

    population_layer.add_to(m)
    colors.add_to(m)

    # Stations Citiz
    citiz_layer = folium.FeatureGroup(
        name="Stations Citiz",
        show=True
    )

    cluster = MarkerCluster(
        name="Citiz",
        disableClusteringAtZoom=13
    ).add_to(citiz_layer)

    for row in stations.itertuples():

        station_name = escape(
            str(row.station_name or "Station Citiz")
        )

        operator = escape(
            str(row.operator or "Citiz")
        )

        popup = (
            f"<b>{station_name}</b>"
            f"<p>Opérateur : {operator}</p>"
        )

        folium.CircleMarker(
            location=[
                float(row.latitude),
                float(row.longitude)
            ],
            radius=6,
            color="#2563eb",
            fill=True,
            fill_color="#2563eb",
            fill_opacity=0.9,
            tooltip=station_name,
            popup=folium.Popup(
                popup, max_width=280
            )
        ).add_to(cluster)

    citiz_layer.add_to(m)

    folium.LayerControl(
        collapsed=False
    ).add_to(m)

    output_file = OUT / cfg["filename"]
    m.save(str(output_file))

    maps[territory] = m

    print("Hexagones :", len(pop))
    print("Stations Citiz :", len(stations))
    print("Fichier :", output_file)
    print(
        "Taille (Mo) :",
        round(output_file.stat().st_size / 1048576, 2)
    )

print("\n=== CARTES GÉNÉRÉES ===")
for territory, cfg in CONFIG.items():
    print(territory, ":", cfg["filename"])

# Première visualisation interactive dans le notebook
from IPython.display import display as ipy_display

ipy_display(maps["METROPOLE"])


In [15]:
from IPython.display import display

display(maps["METROPOLE"])

In [19]:
from IPython.display import display

display(maps["METROPOLE"])
display(maps["MARTINIQUE"])
display(maps["REUNION"])

In [1]:
import h3
import folium
import pandas as pd

print("H3 :", h3.__version__)
print("Folium :", folium.__version__)
print("Bibliothèques OK")

In [2]:
%pip install h3 folium branca

In [3]:
import h3
import folium
import pandas as pd

print("H3 :", h3.__version__)
print("Folium :", folium.__version__)
print("Bibliothèques OK ✅")

In [4]:
from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_25_carte_multimodale_fabric.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError(
        "Le script n'est pas accessible depuis ce Notebook"
    )

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)

In [5]:

from pyspark.sql import functions as F

df = spark.table("silver.france_mobility_map_h3")

display(
    df.filter(
        F.col("layer").isin(
            "velo",
            "trottinette",
            "scooter_indetermine",
            "voiture"
        )
    )
    .groupBy("territory", "layer")
    .agg(
        F.sum("n").alias("vehicules"),
        F.countDistinct("h3_cell").alias("hexagones")
    )
    .orderBy("territory", "layer")
)


In [6]:

from pathlib import Path

script = Path(
    "/lakehouse/default/Files/"
    "etape_25bis_carte_points_reels.py"
)

print("Script trouvé :", script.exists())

if not script.exists():
    raise FileNotFoundError("Script introuvable")

exec(
    compile(
        script.read_text(encoding="utf-8"),
        str(script),
        "exec"
    )
)


In [ ]:

# Not supported

In [7]:
# Load HTML file content
html_content = None
with open("/lakehouse/default/Files/gold/maps/mobilite_points_reels_metropole.html") as html_file:
    html_content = html_file.read()

displayHTML(html_content)
